## <국립민속박물관-단령 스크래핑>

### 1. 접속 준비 (세션 만들기, CSRF 토큰 만들기)
- 국립민속박물관 검색 기능은 보안을 위해 **CSRF 토큰**이라는 임시 값을 요구
    - 이 요청이 실제로 저 검색 페이지를 열어본 사람이 보낸 게 맞다는 걸 증명하기 위함
- 먼저 검색 페이지에 평범하게 접속(GET)해서, 페이지 안에 숨어있는 CSRF 토큰 값 읽기
- 이때 받은 쿠키(로그인 상태 같은 걸 기억하는 값)와 토큰을 계속 재사용해서 이후 요청 보내기

In [5]:
# 1. 필요한 패키지, 라이브러리 불러오기
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# 공통 함수는 utils.py에 모아두고 여기서 가져다 씀 (01~03 노트북이 다 같은 함수를 공유)
from utils import BASE, get_csrf_token, search_relic_list, get_relic_detail

# 2. URL
SEARCH_LIST_URL = f"{BASE}/user/data/home/101/DataRelicCategoryList.do"
DETAIL_URL = f"{BASE}/user/data/home/101/DataRelicView.do"

# 실제 브라우저처럼 보이도록 User-Agent를 지정해줌 -> 일부 서버는 이게 없으면 차단하기 때문
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

# 3. 쿠키를 자동으로 기억해주는 과정 (쿠키 관리해올 필요 없게)
session = requests.Session()
session.headers.update(HEADERS)

# 4. CSRF 토큰 읽기 (utils.py의 함수 사용, session을 인자로 넘겨줌)
csrf_token = get_csrf_token(session)
print("CSRF 토큰:", csrf_token)

CSRF 토큰: 9ca2d97c-5295-4eb6-9bc3-57f99351183d


### 2. 데이터 수집
- 키워드 기반 수집 -> 첫 페이지 수집 -> 상세 페이지 내용 수집

In [6]:
# 1. 키워드 기반 수집
KEYWORDS = ["단령"]

all_list_items = []
for kw in KEYWORDS:
    all_list_items.extend(search_relic_list(session, csrf_token, kw))

print("검색된 전체 건수(중복 포함):", len(all_list_items))

검색된 전체 건수(중복 포함): 126


In [7]:
# 2. 데이터 프레임 제작
list_df = pd.DataFrame(all_list_items)
list_df = list_df.drop_duplicates(subset="seq").reset_index(drop=True)
print("중복 제거 후 소장품 수:", len(list_df))
list_df.head() 

중복 제거 후 소장품 수: 126


,seq,list_title,searched_keyword
0,PS0100200100102806300000,단령(團領),단령
1,PS0100200100102806400000,단령(團領),단령
2,PS0100200100104555500000,단령(團領),단령
3,PS0100200100107759300000,단령(團領),단령
4,PS0100200100104555300000,단령(團領),단령


In [8]:
# 3. 상세 페이지 열기
details = []
for i, row in list_df.iterrows():
    detail = get_relic_detail(session, row["seq"])
    detail["searched_keyword"] = row["searched_keyword"]
    detail["list_title"] = row["list_title"]
    details.append(detail)
    if (i + 1) % 20 == 0:
        print(f"{i + 1} / {len(list_df)} 건 완료")

print("총", len(details), "건")

20 / 126 건 완료
40 / 126 건 완료
60 / 126 건 완료
80 / 126 건 완료
100 / 126 건 완료
120 / 126 건 완료
총 126 건


### 3. 표로 정리 + 파일 저장

In [9]:
# 1. 데이터프레임 확인
df = pd.DataFrame(details)

# 2. 컬럼 순서 정리
priority_cols = ["searched_keyword", "소장품 명칭", "list_title", "국적/시대", "용도/기능",
                 "크기", "소장품 번호", "내용", "image_url", "detail_url", "seq"]
other_cols = [c for c in df.columns if c not in priority_cols]
ordered_cols = [c for c in priority_cols if c in df.columns] + other_cols
df = df[ordered_cols]


dallyeong = df[df["소장품 명칭"].str.contains("단령", na=False)]
print(dallyeong.shape)
dallyeong.head()

(51, 12)


,searched_keyword,소장품 명칭,list_title,국적/시대,용도/기능,크기,소장품 번호,내용,image_url,detail_url,seq,image_urls
0,단령,단령(團領),단령(團領),한국-조선,의-의류-의례복-남자수의,길이 : 126 품 : 54 화장 : 121,028063,둥근 깃의 남성용 겉옷. 충남 태안군 태안읍 삭선리 남오성(1643~1712년) 묘...,https://www.nfm.go.kr/common/apiimage/relic/83...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100102806300000,https://www.nfm.go.kr/common/apiimage/relic/83...
1,단령,단령(團領),단령(團領),한국-조선,의-의류-의례복-남자수의,화장 : 120.5 품 : 57 길이 : 125,028064,둥근 깃의 남성용 겉옷. 충남 태안군 태안읍 삭선리 남오성(1643~1712) 묘 ...,https://www.nfm.go.kr/common/apiimage/relic/85...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100102806400000,https://www.nfm.go.kr/common/apiimage/relic/85...
2,단령,단령(團領),단령(團領),한국-조선,사회생활-의례생활-상장 / 의-의류-평상복-남자포류,길이 : 131.5 화장 : 125.5 품 : 55,045555,"남성용 포(袍). 이진숭(李鎭嵩, 1702~1756) 묘에서 출토된 보공품(補空品)...",https://www.nfm.go.kr/common/apiimage/relic/83...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100104555500000,https://www.nfm.go.kr/common/apiimage/relic/83...
3,단령,단령(團領),단령(團領),한국-조선,사회생활-의례생활-상장 / 의-의류-의례복-남자수의,품 : 50 길이 : 136 화장 : 118,077593,"남성용 포(袍). 신광헌(申光憲, 1731~1784) 묘(경기도 남양주 별내면 화접...",https://www.nfm.go.kr/common/apiimage/relic/84...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100107759300000,https://www.nfm.go.kr/common/apiimage/relic/84...
4,단령,단령(團領),단령(團領),한국-조선,사회생활-의례생활-상장 / 의-의류-평상복-남자포류,화장 : 125 길이 : 129 품 : 52.5,045553,"남성용 포(袍). 이진숭(李鎭嵩, 1702~1756) 묘에서 출토된 보공품(補空品)...",https://www.nfm.go.kr/common/apiimage/relic/83...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100104555300000,https://www.nfm.go.kr/common/apiimage/relic/83...


In [10]:
# 3. 파일 저장(엑셀)
dallyeong.to_excel("../data/nfm_dallyeong.xlsx", index=False)

# 4. 이미지 저장 (한 유물에 사진이 여러 장이면 -1, -2 ... 붙여서 다 저장함)
dallyeong = dallyeong.reset_index(drop=True) 

for i, row in dallyeong.iterrows():
    seq = row["seq"]
    image_urls = str(row["image_urls"]).split("; ") if pd.notna(row["image_urls"]) else []

    for idx, image_url in enumerate(image_urls):
        suffix = "" if len(image_urls) == 1 else f"-{idx + 1}"  # 여러 장일 때만 -1, -2 ... 붙임
        filepath = f"../image/dallyeong/{seq}{suffix}.jpg"

        resp = session.get(image_url, timeout=30)
        with open(filepath, "wb") as f:
            f.write(resp.content)

    if (i + 1) % 51 == 0:
        print(f"{i + 1} / {len(dallyeong)} 완료")

    time.sleep(0.5)

51 / 51 완료
